In [1]:
"""
Grey Wolf Optimizer (GWO) for Job Scheduling
--------------------------------------------
Problem : Assign N jobs to M identical machines so that the MAKESPAN
          (completion time of the busiest machine) is minimized.
Encoding: Each wolf is a vector of length N with values in [0, M).
          Job j is assigned to machine int(position[j]).
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ---------------------------------------------------------------
# 1. Problem definition
# ---------------------------------------------------------------
def decode(position, n_machines):
    """Convert continuous wolf position into machine indices (discrete)."""
    machines = np.floor(position).astype(int)
    return np.clip(machines, 0, n_machines - 1)


def makespan(position, proc_times, n_machines):
    """Objective function: maximum load over all machines (minimize)."""
    assignment = decode(position, n_machines)
    loads = np.zeros(n_machines)
    for job, machine in enumerate(assignment):
        loads[machine] += proc_times[job]
    return loads.max()


# ---------------------------------------------------------------
# 2. Grey Wolf Optimizer
# ---------------------------------------------------------------
def gwo(objective, dim, lb, ub, n_wolves=30, max_iter=200, seed=None):
    rng = np.random.default_rng(seed)

    # Step 4: initialise population randomly in [lb, ub)
    X = lb + rng.random((n_wolves, dim)) * (ub - lb)

    # Step 5: evaluate fitness
    fitness = np.array([objective(x) for x in X])

    # Step 6: alpha, beta, delta
    order = np.argsort(fitness)
    alpha_pos, alpha_score = X[order[0]].copy(), fitness[order[0]]
    beta_pos = X[order[1]].copy()
    delta_pos = X[order[2]].copy()

    history = [alpha_score]

    for t in range(max_iter):
        # Step 8: a decreases linearly from 2 to 0
        a = 2 - t * (2 / max_iter)

        # Step 7: update every wolf's position
        for i in range(n_wolves):
            new_pos = np.zeros(dim)
            for leader in (alpha_pos, beta_pos, delta_pos):
                r1, r2 = rng.random(dim), rng.random(dim)
                A = 2 * a * r1 - a
                C = 2 * r2
                D = np.abs(C * leader - X[i])
                new_pos += leader - A * D
            X[i] = np.clip(new_pos / 3, lb, ub - 1e-9)  # X(t+1) = (X1+X2+X3)/3

        # Step 5 again: re-evaluate fitness
        fitness = np.array([objective(x) for x in X])

        # Step 6 again: update leaders (keep best-so-far alpha)
        order = np.argsort(fitness)
        if fitness[order[0]] < alpha_score:
            alpha_score = fitness[order[0]]
            alpha_pos = X[order[0]].copy()
        beta_pos = X[order[1]].copy()
        delta_pos = X[order[2]].copy()

        history.append(alpha_score)

    # Step 10: return best solution
    return alpha_pos, alpha_score, history


# ---------------------------------------------------------------
# 3. Reporting helpers
# ---------------------------------------------------------------
def print_schedule(assignment, proc_times, n_machines):
    print("\n----- Final Schedule -----")
    for m in range(n_machines):
        jobs = [j for j, mc in enumerate(assignment) if mc == m]
        load = sum(proc_times[j] for j in jobs)
        job_str = ", ".join(f"J{j + 1}({proc_times[j]})" for j in jobs)
        print(f"Machine {m + 1}: [{job_str}]  -> Load = {load}")


def plot_gantt(assignment, proc_times, n_machines, filename="gantt_chart.png"):
    fig, ax = plt.subplots(figsize=(10, 4))
    start_time = np.zeros(n_machines)
    for job, m in enumerate(assignment):
        ax.barh(m, proc_times[job], left=start_time[m],
                edgecolor="black", alpha=0.8)
        ax.text(start_time[m] + proc_times[job] / 2, m, f"J{job + 1}",
                ha="center", va="center", fontsize=8)
        start_time[m] += proc_times[job]
    ax.set_yticks(range(n_machines))
    ax.set_yticklabels([f"M{m + 1}" for m in range(n_machines)])
    ax.set_xlabel("Time")
    ax.set_title("Job Schedule (Gantt Chart)")
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()


def plot_convergence(history, filename="convergence.png"):
    plt.figure(figsize=(7, 4))
    plt.plot(history, linewidth=2)
    plt.xlabel("Iteration")
    plt.ylabel("Best makespan (Alpha)")
    plt.title("GWO Convergence Curve")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()


# ---------------------------------------------------------------
# 4. Main
# ---------------------------------------------------------------
if __name__ == "__main__":
    # Problem data (change these as needed)
    proc_times = np.array([12, 7, 15, 9, 4, 18, 10, 6, 14, 8,
                           11, 5, 16, 3, 13, 9, 7, 12, 10, 6])
    n_jobs = len(proc_times)
    n_machines = 4

    # GWO parameters
    N_WOLVES = 30
    MAX_ITER = 200

    objective = lambda x: makespan(x, proc_times, n_machines)

    best_pos, best_score, history = gwo(
        objective, dim=n_jobs, lb=0, ub=n_machines,
        n_wolves=N_WOLVES, max_iter=MAX_ITER, seed=42
    )

    assignment = decode(best_pos, n_machines)

    print("Jobs            :", n_jobs)
    print("Machines        :", n_machines)
    print("Processing times:", proc_times.tolist())
    print_schedule(assignment, proc_times, n_machines)

    lower_bound = max(np.ceil(proc_times.sum() / n_machines), proc_times.max())
    print(f"\nBest Makespan (Alpha fitness): {best_score}")
    print(f"Theoretical lower bound      : {lower_bound}")

    plot_gantt(assignment, proc_times, n_machines)
    plot_convergence(history)
    print("\nSaved: gantt_chart.png, convergence.png")

Jobs            : 20
Machines        : 4
Processing times: [12, 7, 15, 9, 4, 18, 10, 6, 14, 8, 11, 5, 16, 3, 13, 9, 7, 12, 10, 6]

----- Final Schedule -----
Machine 1: [J2(7), J5(4), J10(8), J11(11), J12(5), J15(13)]  -> Load = 48
Machine 2: [J3(15), J4(9), J13(16), J14(3), J20(6)]  -> Load = 49
Machine 3: [J1(12), J7(10), J16(9), J17(7), J19(10)]  -> Load = 48
Machine 4: [J6(18), J8(6), J9(14), J18(12)]  -> Load = 50

Best Makespan (Alpha fitness): 50.0
Theoretical lower bound      : 49.0

Saved: gantt_chart.png, convergence.png
